## Hello everyone!
You can use this notebook as a starter to your competetion.<br>I'll try to explain every step for begginers<br>Best of luck guys!

## Import libraries:

In [ ]:
import numpy as np
import pandas as pd 
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from catboost import Pool, CatBoostClassifier
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LinearRegression
from scipy.special import expit as sigmoid  # Logistic function
from sklearn.impute import KNNImputer
import xgboost as xgb




from sklearn.metrics import f1_score
pd.set_option('display.max_columns', None)

## Reading Data: 

In [ ]:
train = pd.read_csv("/kaggle/input/widsdatathon2024-challenge1/training.csv")
test = pd.read_csv("/kaggle/input/widsdatathon2024-challenge1/test.csv")
sample_submission = pd.read_csv("/kaggle/input/widsdatathon2024-challenge1/sample_submission.csv")

## Exploring data:

In [ ]:
print(f"The data consists of {train.shape[0]} rows, and {train.shape[1]} columns")

In [ ]:
#This code calculate the percentage of missing data per every row.
for column in train.columns:
    null_count = train[column].isnull().sum()
    if null_count > 0:
        percent_nulls = (null_count / train.shape[0]) * 100
        print(f'{column} : {percent_nulls:.2f}% nulls')

Based on these values I choose to drop columns with the highest null (`Patient_race`, `Payer_type`, `bmi`,`metastatic_first_novel_treatment`, and `metastatic_first_novel_treatment_type`)

`Note:` Any transformation applied to the training data should also be applied to the test data.

In [ ]:
#Dropping high nulls columns
for column in train.columns:
    null_count = train[column].isnull().sum()
    percent_nulls = (null_count / train.shape[0]) * 100
    if percent_nulls > 1:
        train.drop(columns = column,axis = 1,inplace=True)
        test.drop(columns = column,axis = 1,inplace=True)

In [ ]:
#checking that test and train has the same shape
print(f"Train data: {train.shape} \nTest data: {test.shape}")

Everything is good here.

## Handle Missing Data:

In [ ]:
categorical_cols = train.select_dtypes(include=['object', 'category']).columns
numerical_cols = train.select_dtypes(include=['float64', 'int64']).columns

# Impute categorical columns using mode
for col in categorical_cols:
    if col != 'DiagPeriodL90D':
        mode = train[col].mode()[0]
        train[col].fillna(mode, inplace=True)
        test[col].fillna(mode, inplace=True)

# Impute numerical columns using mean
for col in numerical_cols:
    if col != 'DiagPeriodL90D':
        mean = train[col].mean()
        train[col].fillna(mean, inplace=True)
        test[col].fillna(mean, inplace=True)

In [ ]:
#checking that test and train has the same shape
print(f"Train data: {train.shape} \nTest data: {test.shape}")

# label encoder:

In [ ]:
categorical_cols = train.select_dtypes(include=['object', 'category']).columns

le = LabelEncoder()

# Apply LabelEncoder to each categorical column
for col in categorical_cols:
    train[col] = le.fit_transform(train[col])
    test[col] = le.fit_transform(test[col])

In [ ]:
# cols_to_be_dropped = train.corr()['DiagPeriodL90D'].sort_values(ascending=False).tail(5).index
# train.drop(columns =cols_to_be_dropped,inplace=True )
# test.drop(columns =cols_to_be_dropped,inplace=True )

# X & Y

In [ ]:
X = train.drop('DiagPeriodL90D', axis=1)  
y = train['DiagPeriodL90D'] 

## Split into training and testing:

In [ ]:
X_train, X_val, y_train,y_val= train_test_split(X, y, test_size=0.2, random_state=42)

# Pooling:

In [ ]:
train_pool = Pool(X_train,y_train)
val_pool= Pool(X_val,y_val)
test_pool = Pool(test)

## Train the model:

In [ ]:
cat_1 = CatBoostClassifier(iterations=5000, verbose=200,learning_rate = 0.2)
cat_1.fit(train_pool, eval_set=val_pool)
y_pred_catboost_1 = cat_1.predict(val_pool)

f1_score_catboost_1 = f1_score(y_val, y_pred_catboost_1)
print(f'F1 Score (CatBoost): {f1_score_catboost_1}')

In [ ]:
xgb = xgb.XGBClassifier(objective ='binary:logistic', colsample_bytree = 0.3, learning_rate = 0.1,
                max_depth = 5, alpha = 10)

xgb.fit(X_train, y_train)

# Predict
y_pred = xgb.predict(X_val)

# Evaluate the model
# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy * 100:.2f}%")


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

# Predicting
predictions = model.predict(test)

# Transforming predictions to [0, 1] scale
linear_pred = sigmoid(predictions)

# prob_predictions can be interpreted as probabilities
print(linear_pred)

In [ ]:
cat_2 = CatBoostClassifier(iterations=2000, verbose=200, learning_rate = 0.06)
cat_2.fit(train_pool, eval_set=val_pool)
y_pred_catboost_2 = cat_2.predict(val_pool)

f1_score_catboost_2 = f1_score(y_val, y_pred_catboost_2)
print(f'F1 Score (CatBoost): {f1_score_catboost_2}')

## Ensemble:

In [ ]:
y_pred_1 = cat_1.predict_proba(test_pool)[:, 1]
y_pred_2 = cat_2.predict_proba(test_pool)[:, 1]
prediction = 0.4 * y_pred_1 + 0.4 * y_pred_2 + 0.2 *linear_pred
prediction

In [ ]:
v1_sub = pd.read_csv('/kaggle/input/v1-sub/Submission_V1.csv')['DiagPeriodL90D']
sub_2 = pd.read_csv('/kaggle/input/new-sub/submission (6).csv')['DiagPeriodL90D']
test = pd.read_csv("/kaggle/input/widsdatathon2024-challenge1/test.csv")


In [ ]:
prediction = np.resize(prediction,[5792,1]) * 0.1 + np.resize(sub_2,[5792,1]) * 0.55 + np.resize(v1_sub,[5792,1]) * 0.35

In [ ]:
prediction

## Submission:

In [ ]:
test.set_index('patient_id', inplace=True)
test['DiagPeriodL90D'] = prediction

submission = test['DiagPeriodL90D']

submission

In [ ]:
submission.to_csv('submission2.csv')